In [ ]:
!pip -q install -U kagglehub lightgbm

In [ ]:
import kagglehub

kagglehub.login()

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

path = kagglehub.competition_download(
    "energy-anomaly-detection",
    path="train.csv"
)

# Dönen yol dosya veya klasör olursa ikisini de karşılar.
download_path = Path(path)

if download_path.is_file():
    train_path = download_path
else:
    candidates = list(download_path.rglob("train.csv"))
    if not candidates:
        raise FileNotFoundError("İndirilen klasörde train.csv bulunamadı.")
    train_path = candidates[0]

df = pd.read_csv(
    train_path,
    usecols=["building_id", "timestamp", "meter_reading", "anomaly"]
)

print("Veri boyutu:", df.shape)
display(df.head())
display(df.isna().sum().rename("Eksik değer"))
display(df["anomaly"].value_counts(dropna=False).rename("Etiket sayısı"))

UnauthenticatedError: User is not authenticated

In [ ]:
initial_rows = len(df)
cleaning_log = []

def record_step(name, before, after):
    cleaning_log.append({
        "İşlem": name,
        "Silinen satır": before - after
    })

# Veri tiplerini düzelt.
df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")

for col in ["building_id", "meter_reading", "anomaly"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df = df.replace([np.inf, -np.inf], np.nan)

# Kimliği, zamanı, tüketimi veya geçerli etiketi olmayanları çıkar.
before = len(df)
df = df.dropna(subset=["building_id", "timestamp", "meter_reading"])
df = df[df["anomaly"].isin([0, 1])].copy()
record_step("Eksik/geçersiz temel alanlar", before, len(df))

# Negatif tüketimi bu başlangıç çalışmasında geçersiz kabul et.
before = len(df)
df = df[df["meter_reading"] >= 0].copy()
record_step("Negatif tüketim", before, len(df))

# Tamamen aynı kayıtları kaldır.
before = len(df)
df = df.drop_duplicates()
record_step("Tam tekrarlar", before, len(df))

# Aynı bina ve saatte farklı kayıtlar varsa keyfî birini seçme.
before = len(df)
conflicting = df.duplicated(["building_id", "timestamp"], keep=False)
df = df.loc[~conflicting].copy()
record_step("Çakışan bina-zaman kayıtları", before, len(df))

df["building_id"] = df["building_id"].astype("int32")
df["meter_reading"] = df["meter_reading"].astype("float32")
df["anomaly"] = df["anomaly"].astype("int8")

df = df.sort_values(["building_id", "timestamp"]).reset_index(drop=True)

display(pd.DataFrame(cleaning_log))
print(f"Başlangıç: {initial_rows:,}")
print(f"Kalan:     {len(df):,}")
print(f"Silinen:   {initial_rows - len(df):,}")
print(f"Bina:      {df['building_id'].nunique()}")
print(f"Anomali oranı: %{100 * df['anomaly'].mean():.2f}")

assert len(df) > 0, "Temizlik sonrası veri kalmadı."
assert df["anomaly"].nunique() == 2, "İki sınıf da bulunmalı."

In [ ]:
def make_features(data):
    data = data.copy()
    t = data["timestamp"]

    X = pd.DataFrame(index=data.index)

    X["meter_reading"] = data["meter_reading"]
    X["log_reading"] = np.log1p(data["meter_reading"])
    X["is_zero"] = (data["meter_reading"] == 0).astype("int8")

    X["hour"] = t.dt.hour
    X["day_of_week"] = t.dt.dayofweek
    X["month"] = t.dt.month
    X["is_weekend"] = (t.dt.dayofweek >= 5).astype("int8")

    X["hour_sin"] = np.sin(2 * np.pi * X["hour"] / 24)
    X["hour_cos"] = np.cos(2 * np.pi * X["hour"] / 24)

    # Tam olarak ilgili saatlerdeki geçmiş ölçümleri bul.
    lookup = data.set_index(
        ["building_id", "timestamp"]
    )["meter_reading"]

    for hours in [1, 24, 168]:
        previous_keys = pd.MultiIndex.from_arrays(
            [
                data["building_id"].to_numpy(),
                (t - pd.Timedelta(hours=hours)).to_numpy()
            ],
            names=["building_id", "timestamp"]
        )

        past = lookup.reindex(previous_keys).to_numpy()

        X[f"lag_{hours}h"] = past
        X[f"log_diff_{hours}h"] = (
            X["log_reading"].to_numpy() - np.log1p(past)
        )
        X[f"lag_{hours}h_missing"] = np.isnan(past).astype("int8")

    return X.replace([np.inf, -np.inf], np.nan).astype("float32")

X = make_features(df)
y = df["anomaly"]

print("Özellik sayısı:", X.shape[1])
display(X.head())

In [ ]:
from sklearn.model_selection import GroupShuffleSplit

# Önce %20 test binasını ayır.
split_test = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

trainval_idx, test_idx = next(
    split_test.split(X, y, groups=df["building_id"])
)

# Kalan %80'in %25'i, toplamın yaklaşık %20'sidir.
split_val = GroupShuffleSplit(
    n_splits=1,
    test_size=0.25,
    random_state=43
)

train_rel, val_rel = next(
    split_val.split(
        X.iloc[trainval_idx],
        y.iloc[trainval_idx],
        groups=df["building_id"].iloc[trainval_idx]
    )
)

train_idx = trainval_idx[train_rel]
val_idx = trainval_idx[val_rel]

X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]
X_test, y_test = X.iloc[test_idx], y.iloc[test_idx]

building_sets = [
    set(df.iloc[idx]["building_id"])
    for idx in [train_idx, val_idx, test_idx]
]

assert building_sets[0].isdisjoint(building_sets[1])
assert building_sets[0].isdisjoint(building_sets[2])
assert building_sets[1].isdisjoint(building_sets[2])

for name, idx in [
    ("Eğitim", train_idx),
    ("Doğrulama", val_idx),
    ("Test", test_idx)
]:
    labels = y.iloc[idx]
    assert labels.nunique() == 2, f"{name} bölümünde tek sınıf var."

    print(
        f"{name}: {len(idx):,} satır | "
        f"{df.iloc[idx]['building_id'].nunique()} bina | "
        f"Anomali: %{100 * labels.mean():.2f}"
    )

In [ ]:
import lightgbm as lgb

negative_count = int((y_train == 0).sum())
positive_count = int((y_train == 1).sum())

model = lgb.LGBMClassifier(
    objective="binary",
    metric="auc",
    n_estimators=800,
    learning_rate=0.05,
    num_leaves=31,
    max_depth=-1,
    min_child_samples=100,
    colsample_bytree=0.9,
    reg_lambda=1.0,
    scale_pos_weight=negative_count / positive_count,
    random_state=42,
    n_jobs=2,
    verbosity=-1
)

model.fit(
    X_train,
    y_train,
    eval_set=[(X_val, y_val)],
    eval_metric="auc",
    callbacks=[
        lgb.early_stopping(50, first_metric_only=True),
        lgb.log_evaluation(100)
    ]
)

print("Seçilen ağaç sayısı:", model.best_iteration_)

In [ ]:
from sklearn.metrics import precision_recall_curve

val_probability = model.predict_proba(X_val)[:, 1]

precision, recall, thresholds = precision_recall_curve(
    y_val,
    val_probability
)

f1_values = (
    2 * precision[:-1] * recall[:-1]
    / (precision[:-1] + recall[:-1] + 1e-12)
)

best_index = int(np.argmax(f1_values))
best_threshold = float(thresholds[best_index])

print(f"Seçilen eşik: {best_threshold:.4f}")
print(f"Doğrulama F1: {f1_values[best_index]:.4f}")

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    ConfusionMatrixDisplay
)
import matplotlib.pyplot as plt

test_probability = model.predict_proba(X_test)[:, 1]
test_prediction = (test_probability >= best_threshold).astype("int8")

results = pd.DataFrame([{
    "Model": "LightGBM",
    "Accuracy": accuracy_score(y_test, test_prediction),
    "Precision": precision_score(
        y_test, test_prediction, zero_division=0
    ),
    "Recall": recall_score(
        y_test, test_prediction, zero_division=0
    ),
    "F1": f1_score(y_test, test_prediction, zero_division=0),
    "AUROC": roc_auc_score(y_test, test_probability),
    "Average precision": average_precision_score(
        y_test, test_probability
    )
}])

display(results.round(4))

print(
    "Her kayda normal diyen modelin accuracy değeri:",
    round(float((y_test == 0).mean()), 4)
)

ConfusionMatrixDisplay.from_predictions(
    y_test,
    test_prediction,
    display_labels=["Normal", "Anomali"],
    cmap="Blues",
    values_format="d"
)
plt.title("Görülmemiş Binalar — Test Sonuçları")
plt.show()

In [ ]:
import joblib

results.to_csv("/content/model_sonuclari.csv", index=False)

predictions = df.iloc[test_idx][
    ["building_id", "timestamp", "meter_reading", "anomaly"]
].copy()

predictions["anomaly_score"] = test_probability
predictions["prediction"] = test_prediction

predictions.to_csv("/content/test_tahminleri.csv", index=False)

joblib.dump(
    {
        "model": model,
        "threshold": best_threshold,
        "feature_columns": X.columns.tolist()
    },
    "/content/lightgbm_anomali_modeli.joblib"
)

# Özellik üretme fonksiyonunun bulunduğu Colab defterini de sakla.
print("Sonuçlar ve model /content klasörüne kaydedildi.")